# 🧪 Day 1 Lab: Build Your First RAG Pipeline (From Scratch!)

**Welcome to the lab!** 🎉

Alright, you just survived 3 hours of theory. Now it's time to get your hands dirty.

In this lab, you're going to **build a working RAG pipeline step by step** — from raw text all the way to retrieving relevant chunks using similarity search.

### Rules of the Game:
- Each exercise tells you **what to do** — but **you** write the code.
- Don't overthink it. If you're stuck for more than 5 minutes, ask!
- Google and docs are your friends. This isn't an exam, it's practice.
- Have fun with it. Seriously.

### What You'll Need:
- Python 3.8+
- The packages we'll install below
- Your brain (and maybe some coffee ☕)

---

## 🔧 Setup — Install Everything

Run this cell first. Go grab a coffee while it installs. ☕

In [1]:
!pip install -q langchain-text-splitters sentence-transformers chromadb numpy scikit-learn tiktoken


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


---

## 📄 Our Sample Document

Every RAG system starts with documents. Here's ours — a fake (but realistic) university policy document.

**Don't change this cell** — just run it. This is the "knowledge base" your RAG system will search through.

In [3]:
UNIVERSITY_POLICY = """
# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.

Excused absences include documented medical emergencies, official university activities, and bereavement leave (up to 3 days for immediate family members). Students must submit supporting documentation to the Student Affairs office within 5 business days of the absence.

Instructors are responsible for recording attendance at the beginning of each session. Late arrivals (more than 15 minutes after the scheduled start time) will be recorded as half-absences. Students who arrive more than 30 minutes late will be marked as fully absent.

## Chapter 2: Grading System

The university uses a letter grading system based on the following scale: A+ (95-100%), A (90-94%), B+ (85-89%), B (80-84%), C+ (75-79%), C (70-74%), D+ (65-69%), D (60-64%), F (below 60%). A minimum grade of C is required to pass any course.

Grade Point Average (GPA) is calculated on a 4.0 scale where A+ and A both equal 4.0, B+ equals 3.5, B equals 3.0, C+ equals 2.5, C equals 2.0, D+ equals 1.5, D equals 1.0, and F equals 0.0. Students must maintain a cumulative GPA of 2.0 or higher to remain in good academic standing.

Students who wish to dispute a grade must file a formal Grade Appeal within 14 calendar days of the grade being posted. The appeal must be submitted in writing to the department head, including specific reasons why the student believes the grade is incorrect. The department will form a review committee of three faculty members to evaluate the appeal within 30 days.

## Chapter 3: Examination Rules

Final examinations account for no more than 40% of the total course grade. Midterm examinations account for no more than 25%. The remaining percentage must come from continuous assessment components such as assignments, projects, quizzes, and class participation.

Students must bring a valid university ID card to all examinations. Electronic devices including smartphones, smartwatches, and wireless earbuds are strictly prohibited in the examination hall. Possession of any unauthorized electronic device during an exam will be treated as an academic integrity violation, regardless of whether the device was used.

Make-up examinations are only available for students with documented excused absences. Requests for make-up exams must be submitted within 48 hours of the original exam date. The make-up exam may differ in format and content from the original examination.

## Chapter 4: Academic Integrity

The university maintains a zero-tolerance policy toward plagiarism and academic dishonesty. Plagiarism is defined as presenting someone else's work, ideas, or words as your own without proper attribution. This includes copying from published sources, other students' work, or AI-generated content without proper citation.

First-time offenders will receive a zero on the affected assignment and a formal warning. Second-time offenders will receive an F in the course. Third-time offenders face permanent expulsion from the university. All academic integrity violations are permanently recorded in the student's academic file.

Use of AI tools such as ChatGPT is permitted for research and learning purposes but is strictly prohibited for generating submitted coursework unless the instructor explicitly allows it. Students must disclose any AI assistance used in their work.

## Chapter 5: Student Services and Support

The university provides free tutoring services through the Academic Support Center, located in Building 7, Room 201. Tutoring is available for all core subjects Monday through Thursday from 9:00 AM to 5:00 PM and Fridays from 9:00 AM to 1:00 PM.

Students experiencing mental health difficulties can access free counseling services at the Wellness Center. Appointments can be booked online through the student portal or by calling extension 4455. Emergency walk-in appointments are available during business hours.

The Career Development Office offers resume reviews, mock interviews, and internship placement assistance. Students in their third year and above are eligible for the university's industry partnership program, which guarantees at least one internship interview per semester.

## Chapter 6: Library and Research Facilities

The main library is open from 8:00 AM to 10:00 PM on weekdays and from 10:00 AM to 6:00 PM on weekends. During examination periods, the library extends its hours to midnight on weekdays. Students can borrow up to 10 books at a time with a standard loan period of 14 days. Overdue fines are 2 EGP per day per book, with a maximum fine of 50 EGP per book.

Research databases including IEEE Xplore, SpringerLink, and ScienceDirect are accessible through the university network or VPN. Students can request interlibrary loans for materials not available in the university collection, with a typical processing time of 5-7 business days.

## Chapter 7: Financial Policies

Tuition fees must be paid in full before the start of each semester or through the approved installment plan. The installment plan allows payment in three equal installments due at the beginning, middle, and end of the semester. A late payment fee of 5% will be applied to any overdue installment.

Students who withdraw from a course within the first two weeks of the semester are eligible for a full tuition refund for that course. Withdrawal between weeks 2 and 4 results in a 50% refund. No refund is available after week 4. Scholarship students who withdraw may lose their scholarship for the following semester.
"""

print(f"Document loaded! Length: {len(UNIVERSITY_POLICY)} characters")
print(f"That's roughly {len(UNIVERSITY_POLICY) // 4} tokens (rule of thumb: 1 token ≈ 4 chars)")

Document loaded! Length: 5890 characters
That's roughly 1472 tokens (rule of thumb: 1 token ≈ 4 chars)


---

# Exercise 1: Get to Know Your Document 🔍

Before we do anything fancy, let's actually **look** at what we're working with.

A good RAG engineer always inspects the data first!

### 1.1 — How many chapters does this document have?

Write code to count how many times `"## Chapter"` appears in the text.

*(Yeah, it's simple. But in real life, knowing the structure of your documents is step zero.)*

In [4]:
# YOUR CODE HERE
chapter_count = UNIVERSITY_POLICY.count("## Chapter")
print(f"Total chapters: {chapter_count}")

Total chapters: 7


### 1.2 — Extract the chapter titles

Get a list of all the chapter titles (the lines that start with `## Chapter`).

Print them nicely.

In [5]:
# YOUR CODE HERE
chapter_titles = [
    line.strip()
    for line in UNIVERSITY_POLICY.splitlines()
    if line.startswith("## Chapter")
]

for title in chapter_titles:
    print(title)

## Chapter 1: Attendance Policy
## Chapter 2: Grading System
## Chapter 3: Examination Rules
## Chapter 4: Academic Integrity
## Chapter 5: Student Services and Support
## Chapter 6: Library and Research Facilities
## Chapter 7: Financial Policies


### 1.3 — Why can't we just paste this entire document into an LLM?

This is a **thinking question** (no code needed). Write your answer in the markdown cell below.

Think about: context window limits, cost, precision, "lost in the middle" problem...

Give at least **3 reasons**.

*Your answer here:*

1. Context Window & Cost Scalability: While this small sample fits easily, production knowledge bases contain thousands of pages. Sending large contexts significantly increases token latency, API inference costs, and rapidly exceeds model context limits.
2. Lost in the Middle" Effect: Large Language Models recall information best at the very beginning and end of long prompts. Crucial details buried in the middle of massive inputs are frequently overlooked.
3. Retrieval Precision & Noise: Excessive irrelevant text increases distraction, leading to prompt dilution, higher latency, and higher hallucination rates compared to passing only the relevant chunks.

---

# Exercise 2: Text Preprocessing 🧹

Remember from the presentation: preprocessing is **task-dependent**, not a fixed checklist!

Let's see what kind of cleaning makes sense for our university policy document.

### 2.1 — Simulate a "messy" document

Here's a chunk of text that looks like it was badly extracted from a PDF. Your job is to **clean it up**.

Rules:
- Fix the extra whitespace and weird line breaks
- Remove the repeated header/footer junk
- But **keep** the section title and the actual content
- Keep numbers and percentages — they matter!

In [7]:
messy_text = """
University  Policy    Document — Page 14       CONFIDENTIAL


## Chapter 2:     Grading     System


The    university uses     a letter   grading system     based on
the following    scale:  A+    (95-100%),   A (90-94%),
B+ (85-89%),   B   (80-84%),    C+ (75-79%),
C   (70-74%),   D+    (65-69%),   D   (60-64%),
F   (below    60%).


University  Policy    Document — Page 14       CONFIDENTIAL
"""

# YOUR CODE HERE — clean this text
# Hint: you can use .replace(), .strip(), re.sub(), or whatever you want

import re
# 1. Remove repeated header/footer artifacts
cleaned = re.sub(
    r"University\s+Policy\s+Document\s+—\s+Page\s+\d+\s+CONFIDENTIAL",
    "",
    messy_text,
)

# 2. Normalize horizontal whitespace (turn multiple spaces/tabs into a single space)
cleaned = re.sub(r"[^\S\r\n]+", " ", cleaned)

# 3. Normalize multiple line breaks to max 2 newlines
cleaned = re.sub(r"\n\s*\n+", "\n\n", cleaned).strip()

cleaned_text = cleaned  # <-- replace this with your cleaning code

print("=== BEFORE ===")
print(messy_text)
print("\n=== AFTER ===")
print(cleaned_text)

=== BEFORE ===

University  Policy    Document — Page 14       CONFIDENTIAL


## Chapter 2:     Grading     System


The    university uses     a letter   grading system     based on
the following    scale:  A+    (95-100%),   A (90-94%),
B+ (85-89%),   B   (80-84%),    C+ (75-79%),
C   (70-74%),   D+    (65-69%),   D   (60-64%),
F   (below    60%).


University  Policy    Document — Page 14       CONFIDENTIAL


=== AFTER ===
## Chapter 2: Grading System

The university uses a letter grading system based on
the following scale: A+ (95-100%), A (90-94%),
B+ (85-89%), B (80-84%), C+ (75-79%),
C (70-74%), D+ (65-69%), D (60-64%),
F (below 60%).


### 2.2 — Thinking question: What should you NOT remove?

Look at our university policy document. If you were preprocessing it for a RAG system, which of the following would you **keep** and which would you **remove**? And **why**?

Fill in the table below:

| Item | Keep or Remove? | Why? |
|------|----------------|------|
| Chapter headings ("## Chapter 1: Attendance") |  | |
| Percentages like "75%" or "60%" | | |
| The version number "Version 3.2" | | |
| Extra blank lines between paragraphs | | |
| Specific room numbers like "Building 7, Room 201" | | |
| Phone extension "4455" | | |

*Your answer here (fill in the table):*

| Item | Keep or Remove? | Why? |
|------|----------------|------|
| Chapter headings | Keep | Essential semantic anchors providing hierarchical context for retrieval and metadata extraction. |
| Percentages | Keep | Crucial factual policy criteria (e.g., passing grades, minimum attendance limits). |
| Version number | Keep | Defines temporal validity and policy precedence across academic years. |
| Extra blank lines | Remove | Whitespace noise that inflates token counts without conveying semantic meaning. |
| Room numbers | Keep | Concrete entities required to answer procedural and navigational queries. |
| Phone extension | Keep | Specific contact data directly answering support inquiries. |

---

# Exercise 3: Chunking — The Fun Part ✂️

Okay this is where it gets interesting. We're going to split our document into chunks using **different strategies** and see how they compare.

Remember: how you chunk your text has a HUGE impact on retrieval quality!

### 3.1 — Fixed-size Character Chunking

Split `UNIVERSITY_POLICY` into chunks of **500 characters** each.

**No overlap** for now. Just chop it up every 500 characters.

Then:
- Print how many chunks you got
- Print the **first 3 chunks** with their character counts
- Look at where the chunks break — do any of them cut a sentence in half?

In [8]:
# YOUR CODE HERE
# Hint: you can do this with pure Python (a for loop or list comprehension)
# chunk_size = 500
# chunks = [text[i:i+chunk_size] for i in range(0, len(text), chunk_size)]
chunk_size = 500
chunks_fixed = [
    UNIVERSITY_POLICY[i : i + chunk_size]
    for i in range(0, len(UNIVERSITY_POLICY), chunk_size)
]

print(f"Total chunks: {len(chunks_fixed)}")
for idx, c in enumerate(chunks_fixed[:3]):
    print(f"\n--- Chunk {idx+1} ({len(c)} chars) ---\n{c}")

Total chunks: 12

--- Chunk 1 (500 chars) ---

# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.

Excused absences include documented medical emergencies, official uni

--- Chunk 2 (500 chars) ---
versity activities, and bereavement leave (up to 3 days for immediate family members). Students must submit supporting documentation to the Student Affairs office within 5 business days of the absence.

Instructors are responsible for recording attendance at the beginning of each session. Late arrivals (more than 15 minutes after the scheduled start time) will be recorded as half-absences. Students who arrive more than 3

### 3.2 — Now add overlap!

Do the same thing but this time with an **overlap of 50 characters**.

- How many chunks do you get now? (should be more than before!)
- Print chunk 2 and chunk 3. Can you see the overlapping text between them?
- Why does overlap help with retrieval? (write a brief answer)

In [9]:
# YOUR CODE HERE
# Hint: step size = chunk_size - overlap
chunk_size = 500
overlap = 50
step = chunk_size - overlap

chunks_overlap = [
    UNIVERSITY_POLICY[i : i + chunk_size]
    for i in range(0, len(UNIVERSITY_POLICY), step)
]

print(f"Total chunks with overlap: {len(chunks_overlap)}")
print(f"\n--- Chunk 2 ---\n{chunks_overlap[1]}")
print(f"\n--- Chunk 3 ---\n{chunks_overlap[2]}")

# Shared substring check:
shared_text = chunks_overlap[1][-overlap:]
print(
    f"\nOverlapping slice verification: '{shared_text}' in Chunk 3: {shared_text in chunks_overlap[2]}"
)

Total chunks with overlap: 14

--- Chunk 2 ---
clude documented medical emergencies, official university activities, and bereavement leave (up to 3 days for immediate family members). Students must submit supporting documentation to the Student Affairs office within 5 business days of the absence.

Instructors are responsible for recording attendance at the beginning of each session. Late arrivals (more than 15 minutes after the scheduled start time) will be recorded as half-absences. Students who arrive more than 30 minutes late will be mar

--- Chunk 3 ---
s who arrive more than 30 minutes late will be marked as fully absent.

## Chapter 2: Grading System

The university uses a letter grading system based on the following scale: A+ (95-100%), A (90-94%), B+ (85-89%), B (80-84%), C+ (75-79%), C (70-74%), D+ (65-69%), D (60-64%), F (below 60%). A minimum grade of C is required to pass any course.

Grade Point Average (GPA) is calculated on a 4.0 scale where A+ and A both equal 4.0, B+ 

### 3.3 — Recursive Chunking with LangChain

Now let's use the **industry standard** — `RecursiveCharacterTextSplitter` from LangChain.

Do the following:
1. Import `RecursiveCharacterTextSplitter` from `langchain_text_splitters`
2. Create a splitter with `chunk_size=500` and `chunk_overlap=50`
3. Split `UNIVERSITY_POLICY` using `.split_text()`
4. Print the number of chunks
5. Print the first 3 chunks

**Compare**: Do these chunks look "smarter" than the fixed-size ones from 3.1? How?

In [10]:
# YOUR CODE HERE
from langchain_text_splitters import RecursiveCharacterTextSplitter

recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""],
)

recursive_chunks = recursive_splitter.split_text(UNIVERSITY_POLICY)
print(f"Total recursive chunks: {len(recursive_chunks)}")

for idx, c in enumerate(recursive_chunks[:3]):
    print(f"\n--- Recursive Chunk {idx+1} ({len(c)} chars) ---\n{c}")

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Total recursive chunks: 19

--- Recursive Chunk 1 (428 chars) ---
# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.

--- Recursive Chunk 2 (270 chars) ---
Excused absences include documented medical emergencies, official university activities, and bereavement leave (up to 3 days for immediate family members). Students must submit supporting documentation to the Student Affairs office within 5 business days of the absence.

--- Recursive Chunk 3 (297 chars) ---
Instructors are responsible for recording attendance at the beginning of each session. Late arrivals (more than 15 minutes after the scheduled start time) w

### 3.4 — Experiment with chunk sizes

Let's see how chunk size affects the number of chunks and their quality.

Using `RecursiveCharacterTextSplitter`, split the document with these **3 different chunk sizes**:
- `chunk_size=200` (small)
- `chunk_size=500` (medium)
- `chunk_size=1000` (large)

Keep `chunk_overlap=50` for all.

For each:
1. Print the total number of chunks
2. Print the **average** chunk length (in characters)
3. Pick one chunk from each and read it — which size gives the most "meaningful" chunks?

In [11]:
# YOUR CODE HERE
# Try all three sizes and compare!
sizes = [200, 500, 1000]

for size in sizes:
    splitter = RecursiveCharacterTextSplitter(chunk_size=size, chunk_overlap=50)
    splits = splitter.split_text(UNIVERSITY_POLICY)
    avg_len = sum(len(s) for s in splits) / len(splits) if splits else 0
    print(
        f"Size: {size:4d} | Chunks: {len(splits):2d} | Avg Length: {avg_len:.1f} chars"
    )
    print(f"Sample preview: {splits[0][:80]}...\n")

Size:  200 | Chunks: 48 | Avg Length: 143.0 chars
Sample preview: # University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

...

Size:  500 | Chunks: 19 | Avg Length: 319.7 chars
Sample preview: # University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

...

Size: 1000 | Chunks:  7 | Avg Length: 871.1 chars
Sample preview: # University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

...



### 3.5 — Thinking question: Which chunk size would you pick?

For our university policy Q&A system, which chunk size do you think works best? Why?

Think about the tradeoff:
- Too small → precise but loses context
- Too large → more context but less precise, more noise

*Your answer here:* Thinking Question: Optimal Chunk Size Choice
For this policy document, 500 characters (~100–125 tokens) works best. 200 characters is too fragmented, frequently separating conditions from consequences (e.g., separating attendance limits from penalties). Conversely, 1000 characters aggregates multiple distinct policy stipulations into one vector, diluting semantic search accuracy.



### 3.6 — Markdown-based / Structure-aware Chunking

Our document has nice Markdown headers (`## Chapter ...`). Let's use them!

Import `MarkdownHeaderTextSplitter` from `langchain_text_splitters` and split the document by its headers.

Use these headers to split on:
```python
headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]
```

- How many chunks do you get?
- Print each chunk's `metadata` and the first 100 characters of its `page_content`
- How is this different from the recursive chunking approach? Is it better for this document?

In [12]:
# YOUR CODE HERE
from langchain_text_splitters import MarkdownHeaderTextSplitter
from langchain_text_splitters import MarkdownHeaderTextSplitter

headers_to_split_on = [
    ("#", "Header 1"),
    ("##", "Header 2"),
]

markdown_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=headers_to_split_on
)
md_chunks = markdown_splitter.split_text(UNIVERSITY_POLICY)

print(f"Total markdown chunks: {len(md_chunks)}")
for idx, chunk in enumerate(md_chunks):
    print(f"\nChunk {idx+1} Metadata: {chunk.metadata}")
    print(f"Content preview: {chunk.page_content[:100]}...")

Total markdown chunks: 7

Chunk 1 Metadata: {'Header 1': 'University Academic Policy Document', 'Header 2': 'Chapter 1: Attendance Policy'}
Content preview: All students are required to attend a minimum of 75% of scheduled classes for each course. Students ...

Chunk 2 Metadata: {'Header 1': 'University Academic Policy Document', 'Header 2': 'Chapter 2: Grading System'}
Content preview: The university uses a letter grading system based on the following scale: A+ (95-100%), A (90-94%), ...

Chunk 3 Metadata: {'Header 1': 'University Academic Policy Document', 'Header 2': 'Chapter 3: Examination Rules'}
Content preview: Final examinations account for no more than 40% of the total course grade. Midterm examinations acco...

Chunk 4 Metadata: {'Header 1': 'University Academic Policy Document', 'Header 2': 'Chapter 4: Academic Integrity'}
Content preview: The university maintains a zero-tolerance policy toward plagiarism and academic dishonesty. Plagiari...

Chunk 5 Metadata: {'Header 1': '

---

# Exercise 4: Embeddings — Turning Text into Numbers 🔢

Now we're getting to the core of RAG. We need to convert our text chunks into **vectors** (embeddings) so we can search them by **meaning**, not just keywords.

We'll use the `sentence-transformers` library — it's free, local, and works great!

### 4.1 — Load an embedding model

Load the `all-MiniLM-L6-v2` model from `sentence_transformers`.

This is a small, fast model that's perfect for learning (and honestly, good enough for many real applications too).

```python
from sentence_transformers import SentenceTransformer
model = SentenceTransformer('all-MiniLM-L6-v2')
```

After loading it, answer these questions:
- What is the **embedding dimension** of this model? (Hint: encode any sentence and check the `.shape`)
- What does that number mean?

In [13]:
# YOUR CODE HERE
from sentence_transformers import SentenceTransformer
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-MiniLM-L6-v2")

query_text = "What is the attendance policy?"
vector = model.encode(query_text)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6529.83it/s]


### 4.2 — Embed a sentence and inspect it

Encode this sentence: `"What is the attendance policy?"`

Then:
1. Print the shape of the resulting vector
2. Print the first 10 values of the vector
3. What do these numbers represent? (just a brief answer — no essay needed!)

In [14]:
# YOUR CODE HERE
# Hint: model.encode("your text here")
print(f"Embedding Shape: {vector.shape}")
print(f"First 10 values: {vector[:10]}")

Embedding Shape: (384,)
First 10 values: [ 0.03555752  0.06524532  0.00655052 -0.0042237   0.02168333  0.08371311
  0.03467272 -0.10015112 -0.03849138  0.0011442 ]


### 4.3 — Semantic similarity experiment 🧪

This is where it gets cool. Let's see if the model actually "understands" meaning!

Encode these 5 sentences:

```python
sentences = [
    "What is the attendance requirement?",        # Query
    "Students must attend 75% of classes.",       # Relevant!
    "The grading scale uses A through F.",        # Different topic
    "Class presence is mandatory for most sessions.",  # Relevant (paraphrase!)
    "The library is open until 10 PM.",           # Totally unrelated
]
```

Then:
1. Compute the **cosine similarity** between the first sentence (the query) and each of the other 4
2. Print the similarity scores
3. Which sentences are most similar to the query? Does this match your intuition?

Hint for cosine similarity:
```python
from sklearn.metrics.pairwise import cosine_similarity
# cosine_similarity(vector_a.reshape(1, -1), vector_b.reshape(1, -1))
```

In [15]:
# YOUR CODE HERE
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

sentences = [
    "What is the attendance requirement?",
    "Students must attend 75% of classes.",
    "The grading scale uses A through F.",
    "Class presence is mandatory for most sessions.",
    "The library is open until 10 PM.",
]

embeddings = model.encode(sentences)
query_vec = embeddings[0].reshape(1, -1)

for i in range(1, len(sentences)):
    score = cosine_similarity(query_vec, embeddings[i].reshape(1, -1))[0][0]
    print(f"Score: {score:.4f} | Sentence: '{sentences[i]}'")

Score: 0.5588 | Sentence: 'Students must attend 75% of classes.'
Score: 0.1907 | Sentence: 'The grading scale uses A through F.'
Score: 0.5005 | Sentence: 'Class presence is mandatory for most sessions.'
Score: 0.1718 | Sentence: 'The library is open until 10 PM.'


### 4.4 — Dense vs Keyword matching

Here's a fun one. Let's see where dense embeddings shine and where they struggle.

Check the similarity between these pairs:

**Pair 1 (Paraphrase — no shared keywords):**
- `"Can I skip class?"` 
- `"Students must attend 75% of classes."`

**Pair 2 (Same keywords — different meaning):**
- `"I need to pass the class"`
- `"I need a hall pass for the class"`

**Pair 3 (Exact term match):**
- `"GPA 2.0 requirement"`
- `"cumulative GPA of 2.0 or higher"`

Compute cosine similarity for each pair. What do you notice? When does semantic search work great, and when might keyword search do better?

In [16]:
# YOUR CODE HERE
pairs = [
    ("Can I skip class?", "Students must attend 75% of classes."),
    ("I need to pass the class", "I need a hall pass for the class"),
    ("GPA 2.0 requirement", "cumulative GPA of 2.0 or higher"),
]

for s1, s2 in pairs:
    v1 = model.encode([s1])
    v2 = model.encode([s2])
    sim = cosine_similarity(v1, v2)[0][0]
    print(f"Sim: {sim:.4f} | '{s1}' <--> '{s2}'")

Sim: 0.3978 | 'Can I skip class?' <--> 'Students must attend 75% of classes.'
Sim: 0.7450 | 'I need to pass the class' <--> 'I need a hall pass for the class'
Sim: 0.7521 | 'GPA 2.0 requirement' <--> 'cumulative GPA of 2.0 or higher'


### 4.5 — Embed ALL the chunks!

Now let's embed our actual document chunks.

1. Use the `RecursiveCharacterTextSplitter` from Exercise 3.3 (chunk_size=500, overlap=50) to get chunks
2. Embed ALL chunks using `model.encode(chunks)`
3. Print the shape of the resulting embedding matrix
4. What does each dimension of this matrix represent? (rows = ?, columns = ?)

In [17]:
# YOUR CODE HERE
# Step 1: chunk the document
# Step 2: embed all chunks
# Step 3: print shape
chunks = recursive_chunks
chunk_embeddings = model.encode(chunks)

print(f"Chunk Embeddings Matrix Shape: {chunk_embeddings.shape}")
# Rows = number of chunks, Columns = 384 embedding dimensions

Chunk Embeddings Matrix Shape: (19, 384)


---

# Exercise 5: Similarity Search — Finding the Right Chunks 🎯

Now that we have chunks and their embeddings, let's do what RAG is all about: **retrieval**!

Given a user question, find the most relevant chunks.

### 5.1 — Manual similarity search

Let's do this **from scratch** first — no fancy libraries. Just numpy and cosine similarity.

Given this query: `"What happens if I miss too many classes?"`

1. Embed the query using the same model
2. Compute the cosine similarity between the query embedding and ALL chunk embeddings
3. Sort the results by similarity score (highest first)
4. Print the **top 3** most relevant chunks with their similarity scores

Does the result make sense? Is the system retrieving chunks about attendance?

In [18]:
# YOUR CODE HERE
query = "What happens if I miss too many classes?"

# Step 1: embed the query
# Step 2: compute cosine similarity with all chunk embeddings
# Step 3: sort and get top 3
# Step 4: print results
query = "What happens if I miss too many classes?"
q_vec = model.encode([query])

similarities = cosine_similarity(q_vec, chunk_embeddings)[0]
top_indices = np.argsort(similarities)[::-1][:3]

print(f"Query: {query}\n")
for rank, idx in enumerate(top_indices, 1):
    print(f"Rank {rank} | Score: {similarities[idx]:.4f}")
    print(f"{chunks[idx]}\n{'-'*50}")

Query: What happens if I miss too many classes?

Rank 1 | Score: 0.5619
# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.
--------------------------------------------------
Rank 2 | Score: 0.4639
Instructors are responsible for recording attendance at the beginning of each session. Late arrivals (more than 15 minutes after the scheduled start time) will be recorded as half-absences. Students who arrive more than 30 minutes late will be marked as fully absent.

## Chapter 2: Grading System
--------------------------------------------------
Rank 3 | Score: 0.3733
Students who withdraw from a course within the first 

### 5.2 — Try different queries!

Test your retrieval system with these queries and print the **top 2** chunks for each:

1. `"How is GPA calculated?"`
2. `"Can I use ChatGPT for my assignments?"`
3. `"Where can I get mental health support?"`
4. `"What is the late payment fee?"`

For each query, check: does the top result actually answer the question? 👀

**Bonus**: Try writing a query that the system gets WRONG. Can you break it?

In [19]:
# YOUR CODE HERE
# Hint: write a function! You'll be doing this a lot.
# def search(query, chunks, chunk_embeddings, model, top_k=2):
#     ...
def search(query, chunks, chunk_embeddings, model, top_k=2):
    q_emb = model.encode([query])
    sims = cosine_similarity(q_emb, chunk_embeddings)[0]
    top_k_idx = np.argsort(sims)[::-1][:top_k]
    return [(idx, sims[idx], chunks[idx]) for idx in top_k_idx]


test_queries = [
    "How is GPA calculated?",
    "Can I use ChatGPT for my assignments?",
    "Where can I get mental health support?",
    "What is the late payment fee?",
]

for q in test_queries:
    print(f"=== Query: '{q}' ===")
    results = search(q, chunks, chunk_embeddings, model, top_k=2)
    for rank, (idx, score, text) in enumerate(results, 1):
        print(f"  [{rank}] Score: {score:.4f} | Preview: {text[:100]}...")
    print()

=== Query: 'How is GPA calculated?' ===
  [1] Score: 0.7210 | Preview: Grade Point Average (GPA) is calculated on a 4.0 scale where A+ and A both equal 4.0, B+ equals 3.5,...
  [2] Score: 0.4463 | Preview: ## Chapter 2: Grading System

The university uses a letter grading system based on the following sca...

=== Query: 'Can I use ChatGPT for my assignments?' ===
  [1] Score: 0.6825 | Preview: Use of AI tools such as ChatGPT is permitted for research and learning purposes but is strictly proh...
  [2] Score: 0.2057 | Preview: ## Chapter 5: Student Services and Support

The university provides free tutoring services through t...

=== Query: 'Where can I get mental health support?' ===
  [1] Score: 0.6366 | Preview: Students experiencing mental health difficulties can access free counseling services at the Wellness...
  [2] Score: 0.1883 | Preview: Excused absences include documented medical emergencies, official university activities, and bereave...

=== Query: 'What is the late payment

### 5.3 — Compare similarity metrics

Let's compare the three metrics we learned about.

For the query `"What is the grading system?"`, compute:
1. **Cosine Similarity** (using `sklearn.metrics.pairwise.cosine_similarity`)
2. **Dot Product** (using `numpy.dot`)
3. **Euclidean Distance** (using `numpy.linalg.norm(a - b)` or `sklearn.metrics.pairwise.euclidean_distances`)

Rank the top 3 chunks using each metric. Do they give the **same ranking** or different ones?

Remember:
- Cosine & Dot Product: **Higher** = more similar
- Euclidean Distance: **Lower** = more similar

In [20]:
# YOUR CODE HERE
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances
from sklearn.metrics.pairwise import euclidean_distances

query_metric = "What is the grading system?"
q_vec = model.encode([query_metric])

# 1. Cosine Similarity
cos_sim = cosine_similarity(q_vec, chunk_embeddings)[0]
top_cos = np.argsort(cos_sim)[::-1][:3]

# 2. Dot Product
dot_prod = np.dot(chunk_embeddings, q_vec.T).squeeze()
top_dot = np.argsort(dot_prod)[::-1][:3]

# 3. Euclidean Distance (L2)
l2_dist = euclidean_distances(q_vec, chunk_embeddings)[0]
top_l2 = np.argsort(l2_dist)[:3]

print("Top 3 Indices by Metric:")
print(f"Cosine Similarity : {top_cos.tolist()}")
print(f"Dot Product       : {top_dot.tolist()}")
print(f"Euclidean Distance: {top_l2.tolist()}")

Top 3 Indices by Metric:
Cosine Similarity : [3, 4, 6]
Dot Product       : [3, 4, 6]
Euclidean Distance: [3, 4, 6]


---

# Exercise 6: Vector Database with ChromaDB 🗄️

Doing similarity search manually with numpy is fine for learning, but in practice we use a **vector database**.

Let's use **ChromaDB** — it's easy to set up and perfect for this lab.

### 6.1 — Create a Chroma collection and add your chunks

1. Import `chromadb` and create an in-memory client
2. Create a collection called `"university_policy"`
3. Add all your chunks to the collection
   - Each chunk needs a unique `id` (e.g., `"chunk_0"`, `"chunk_1"`, ...)
   - Add the chunk text as `documents`
   - Add some metadata for each chunk — at minimum, add `{"chunk_index": i}` for each

Here's a skeleton to get you started:
```python
import chromadb
client = chromadb.Client()  # in-memory
collection = client.create_collection(name="university_policy")

# Add documents...
```

After adding, print: `collection.count()` to verify everything got in.

In [21]:
# YOUR CODE HERE
import chromadb
import chromadb

client = chromadb.Client()
collection = client.get_or_create_collection(name="university_policy")

ids = [f"chunk_{i}" for i in range(len(chunks))]
metadatas = [{"chunk_index": i} for i in range(len(chunks))]

collection.add(documents=chunks, ids=ids, metadatas=metadatas)

print(f"Total chunks added: {collection.count()}")

C:\Users\HP\.cache\chroma\onnx_models\all-MiniLM-L6-v2\onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:31<00:00, 2.66MiB/s]


Total chunks added: 19


### 6.2 — Query the vector database

Now query your collection!

Use `collection.query()` with the following queries and `n_results=3`:

1. `"What is the penalty for cheating?"`
2. `"How do I get a tutor?"`
3. `"Can I get a refund if I drop a course?"`

For each result, print:
- The returned documents
- The distances (similarity scores)

Does ChromaDB give you the same results as your manual search from Exercise 5? 🤔

In [22]:
# YOUR CODE HERE
# Hint: collection.query(query_texts=["your query"], n_results=3)
chroma_queries = [
    "What is the penalty for cheating?",
    "How do I get a tutor?",
    "Can I get a refund if I drop a course?",
]

for q in chroma_queries:
    res = collection.query(query_texts=[q], n_results=3)
    print(f"\n================ Query: {q} ================")
    for doc, dist in zip(res["documents"][0], res["distances"][0]):
        print(f"Distance: {dist:.4f} | Content: {doc[:110]}...")


================ Query: What is the penalty for cheating? ================
Distance: 1.2392 | Content: First-time offenders will receive a zero on the affected assignment and a formal warning. Second-time offender...
Distance: 1.4716 | Content: ## Chapter 4: Academic Integrity

The university maintains a zero-tolerance policy toward plagiarism and acade...
Distance: 1.5260 | Content: Students must bring a valid university ID card to all examinations. Electronic devices including smartphones, ...

================ Query: How do I get a tutor? ================
Distance: 0.5572 | Content: ## Chapter 5: Student Services and Support

The university provides free tutoring services through the Academi...
Distance: 1.4020 | Content: Use of AI tools such as ChatGPT is permitted for research and learning purposes but is strictly prohibited for...
Distance: 1.4193 | Content: Students experiencing mental health difficulties can access free counseling services at the Wellness Center. A...

=======

### 6.3 — Filtering with metadata!

One of the superpowers of vector databases is **metadata filtering**.

Let's make this more useful. Delete the old collection and create a new one where each chunk has richer metadata.

For each chunk, figure out which chapter it belongs to and add `{"chapter": "Chapter X: ...."}` as metadata.

Hint: You could check if the chunk contains certain keywords or use the chapter titles you extracted in Exercise 1.2. Don't overthink it — a simple approach is fine!

Then query with a **metadata filter**:
```python
collection.query(
    query_texts=["What is the late fee?"],
    n_results=3,
    where={"chapter": "Chapter 7: Financial Policies"}
)
```

Compare the results with and without the filter. Does filtering help?

In [23]:
# YOUR CODE HERE
# This one's a bit more involved — take your time!
client.delete_collection(name="university_policy")
collection = client.create_collection(name="university_policy")

# Tag chunks with chapter metadata based on chapter boundaries
enriched_metadatas = []
current_chapter = "General"

for chunk in chunks:
    for title in chapter_titles:
        if title in chunk:
            current_chapter = title.replace("## ", "")
            break
    enriched_metadatas.append(
        {"chapter": current_chapter, "chunk_len": len(chunk)}
    )

collection.add(documents=chunks, ids=ids, metadatas=enriched_metadatas)

# Query with metadata filter
filtered_result = collection.query(
    query_texts=["What is the late fee?"],
    n_results=3,
    where={"chapter": "Chapter 7: Financial Policies"},
)

print("Filtered Results:")
for doc, meta in zip(
    filtered_result["documents"][0], filtered_result["metadatas"][0]
):
    print(f"Metadata: {meta} | Content: {doc[:100]}...")

Filtered Results:
Metadata: {'chunk_len': 331, 'chapter': 'Chapter 7: Financial Policies'} | Content: ## Chapter 7: Financial Policies

Tuition fees must be paid in full before the start of each semeste...
Metadata: {'chunk_len': 318, 'chapter': 'Chapter 7: Financial Policies'} | Content: Students who withdraw from a course within the first two weeks of the semester are eligible for a fu...
Metadata: {'chunk_len': 312, 'chapter': 'Chapter 7: Financial Policies'} | Content: Research databases including IEEE Xplore, SpringerLink, and ScienceDirect are accessible through the...


---

# Exercise 7: Put It All Together — Mini RAG Pipeline! 🚀

Alright, final boss. Let's combine everything into one clean pipeline.

You're NOT going to call an actual LLM (we'll save that for Day 2), but you'll build **everything else**: the indexing pipeline and the retrieval pipeline.

### 7.1 — Build the complete indexing pipeline

Write a function called `build_index` that takes a raw document string and:

1. **Preprocesses** it (basic cleaning — remove extra whitespace, etc.)
2. **Chunks** it using RecursiveCharacterTextSplitter (pick your favorite chunk size)
3. **Stores** the chunks in a ChromaDB collection with metadata
4. Returns the collection

```python
def build_index(document: str, collection_name: str = "rag_index") -> chromadb.Collection:
    # YOUR CODE HERE
    pass
```

In [28]:
# YOUR CODE HERE

def build_index(document, collection_name="rag_index"):
    """Build a searchable index from a raw document."""
      # Replace with your implementation
      # 1. Clean document
    cleaned_doc = re.sub(r"[^\S\r\n]+", " ", document)
    cleaned_doc = re.sub(r"\n\s*\n+", "\n\n", cleaned_doc).strip()

    # 2. Chunking
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=500,
        chunk_overlap=50,
        separators=["\n\n", "\n", " ", ""],
    )
    doc_chunks = splitter.split_text(cleaned_doc)

    # 3. ChromaDB setup
    chroma_client = chromadb.Client()
    try:
        chroma_client.delete_collection(collection_name)
    except Exception:
        pass

    col = chroma_client.create_collection(
        name=collection_name, metadata={"hnsw:space": "cosine"}
    )

    ids = [f"idx_{i}" for i in range(len(doc_chunks))]
    metadatas = [{"chunk_id": i} for i in range(len(doc_chunks))]

    col.add(documents=doc_chunks, ids=ids, metadatas=metadatas)
    return col

# Test it:
collection = build_index(UNIVERSITY_POLICY)
print(f"Index built! Total chunks: {collection.count()}")

Index built! Total chunks: 19


### 7.2 — Build the retrieval function

Write a function called `retrieve` that:

1. Takes a user query and a ChromaDB collection
2. Searches for the top-K most relevant chunks
3. Returns the chunks formatted nicely as a **context string** that you could paste into an LLM prompt

```python
def retrieve(query: str, collection, top_k: int = 3) -> str:
    # YOUR CODE HERE
    pass
```

The output should look something like:
```
[Source 1] (similarity: 0.87)
Students must attend 75% of classes...

[Source 2] (similarity: 0.72)
Excused absences include...
```

In [27]:
# YOUR CODE HERE

def retrieve(query, collection, top_k=3):
    """Retrieve the most relevant chunks for a query."""
    # Replace with your implementation
    results = collection.query(query_texts=[query], n_results=top_k)
    context_blocks = []

    for i, (doc, dist) in enumerate(
        zip(results["documents"][0], results["distances"][0]), start=1
    ):
        # Chroma cosine distance = 1 - cosine_similarity
        sim_score = max(0.0, 1.0 - dist)
        context_blocks.append(
            f"[Source {i}] (similarity: {sim_score:.2f})\n{doc}"
        )

    return "\n\n".join(context_blocks)


# Test it:
result = retrieve("What happens if I cheat on an exam?", collection)
print(result)

[Source 1] (similarity: 0.46)
Students must bring a valid university ID card to all examinations. Electronic devices including smartphones, smartwatches, and wireless earbuds are strictly prohibited in the examination hall. Possession of any unauthorized electronic device during an exam will be treated as an academic integrity violation, regardless of whether the device was used.

[Source 2] (similarity: 0.45)
First-time offenders will receive a zero on the affected assignment and a formal warning. Second-time offenders will receive an F in the course. Third-time offenders face permanent expulsion from the university. All academic integrity violations are permanently recorded in the student's academic file.

[Source 3] (similarity: 0.45)
Make-up examinations are only available for students with documented excused absences. Requests for make-up exams must be submitted within 48 hours of the original exam date. The make-up exam may differ in format and content from the original examinati

### 7.3 — Build a fake "answer" function (prompt template)

We don't have an LLM today, but let's prepare the **prompt** that we WOULD send to one.

Write a function `build_prompt` that:
1. Takes a user query and the retrieved context (from your `retrieve` function)
2. Returns a nicely formatted prompt string

Use this template (or make your own!):

```
You are a helpful university assistant. Answer the student's question based ONLY on the provided context. If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{retrieved_context}

Question: {query}

Answer:
```

Test it with a few queries and print the full prompts. Imagine you're the LLM — could YOU answer the question from the given context?

In [26]:
# YOUR CODE HERE

def build_prompt(query, context):
    """Build a prompt for the LLM."""
    # Replace with your implementation
    template = f"""You are a helpful university assistant. Answer the student's question based ONLY on the provided context. If the answer is not in the context, say "I don't have enough information to answer that."

Context:
{context}

Question: {query}

Answer:"""
    return template


# Test the full pipeline:
# query = "Can I use AI tools for my homework?"
# context = retrieve(query, collection)
# prompt = build_prompt(query, context)
# print(prompt)

### 7.4 — Stress test your pipeline! 🏋️

Run these 6 queries through your complete pipeline (build_index → retrieve → build_prompt).

For each query, check:
- ✅ Did it retrieve the right context?
- ❌ Did it retrieve something irrelevant?
- 🤔 Could an LLM answer correctly from this context?

```python
test_queries = [
    "What percentage of classes do I need to attend?",
    "What grade do I need to pass a course?",
    "Where is the tutoring center located?",
    "What happens if I plagiarize for the second time?",
    "How many books can I borrow from the library?",
    "What is the refund policy if I drop a course after week 3?",
]
```

Write your observations below!

In [29]:
# YOUR CODE HERE
test_queries = [
    "What percentage of classes do I need to attend?",
    "What grade do I need to pass a course?",
    "Where is the tutoring center located?",
    "What happens if I plagiarize for the second time?",
    "How many books can I borrow from the library?",
    "What is the refund policy if I drop a course after week 3?",
]
# Run each query through the pipeline and print results
for query in test_queries:
    context = retrieve(query, collection, top_k=2)
    prompt = build_prompt(query, context)
    print(f"\n{'='*70}\n{prompt}\n")



You are a helpful university assistant. Answer the student's question based ONLY on the provided context. If the answer is not in the context, say "I don't have enough information to answer that."

Context:
[Source 1] (similarity: 0.62)
# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.

[Source 2] (similarity: 0.47)
## Chapter 2: Grading System

The university uses a letter grading system based on the following scale: A+ (95-100%), A (90-94%), B+ (85-89%), B (80-84%), C+ (75-79%), C (70-74%), D+ (65-69%), D (60-64%), F (below 60%). A minimum grade of C is required to pass any course.

Question: What percentage of

*Your observations:*

- Query 1: ✅/❌  ✅ Retrieved Chapter 1 chunk citing the 75% class attendance rule.
- Query 2: ✅/❌  ✅ Retrieved Chapter 2 chunk stating a minimum grade of "C" is required.
- Query 3: ✅/❌  ✅ Retrieved Chapter 5 chunk detailing Academic Support Center in Building 7, Room 201.
- Query 4: ✅/❌  ✅ Retrieved Chapter 4 chunk detailing penalties (failing the course for 2nd offense).
- Query 5: ✅/❌  ✅ Retrieved Chapter 6 chunk stating maximum loan of 10 books.
- Query 6: ✅/❌  ✅ Retrieved Chapter 7 chunk showing 50% refund between weeks 2 and 4.

---

# Bonus Exercise: Break Your Own System! 💥

*(Optional but highly recommended)*

Every RAG system has weaknesses. Let's find yours!

Try to come up with queries that your system handles **badly**. Things like:

- Questions that need information from **multiple chapters** to answer
- Very **vague** questions
- Questions about things that are **NOT** in the document
- Questions with **different wording** than what's in the document
- Questions about **numbers or specific values**

For each failing query, think about: **What would fix this?** (hint: better chunking? bigger chunks? hybrid search? metadata filtering?)

This is actually one of the most important skills in RAG engineering — knowing where your system fails and how to improve it! 🧠

In [30]:
# YOUR CODE HERE — try to break your system!

tricky_queries = [
    # 1. Cross-chapter reasoning
    "Can missing classes affect my scholarship or financial status?",
    # 2. Out-of-domain knowledge
    "Where can I park my car on campus?",
    # 3. Numeric ambiguity
    "What is the policy for three days?",
]

for q in tricky_queries:
    ctx = retrieve(q, collection, top_k=2)
    print(f"TRICKY QUERY: {q}\n{ctx}\n{'-'*50}")


TRICKY QUERY: Can missing classes affect my scholarship or financial status?
[Source 1] (similarity: 0.54)
# University Academic Policy Document
## Version 3.2 — Academic Year 2024-2025

## Chapter 1: Attendance Policy

All students are required to attend a minimum of 75% of scheduled classes for each course. Students who fall below this threshold will receive an official warning after missing 15% of classes. If attendance drops below 60%, the student will be automatically barred from taking the final examination for that course.

[Source 2] (similarity: 0.48)
Students who withdraw from a course within the first two weeks of the semester are eligible for a full tuition refund for that course. Withdrawal between weeks 2 and 4 results in a 50% refund. No refund is available after week 4. Scholarship students who withdraw may lose their scholarship for the following semester.
--------------------------------------------------
TRICKY QUERY: Where can I park my car on campus?
[Source 1] (si

*What queries broke the system and why?*

1. Cross-Chapter Queries: Splitting knowledge across Chapter 1 (Attendance) and Chapter 7 (Financials) means isolated top-K chunks often capture only half the answer. Fix: Hybrid search with multi-query routing or cross-chunk reranking.
2. Out-of-Domain Inquiries: Dense embeddings return the nearest vector even when similarity is objectively poor. Fix: Enforce strict similarity score thresholds to trigger fallback rejections.
3. Short/Vague Queries: "What is the policy for three days?" overlaps bereavement leave (Chapter 1) and installment policies (Chapter 7). Fix: Query expansion and hypothetical document embeddings (HyDE).

---

# 🎉 Lab Complete!

Congrats! You just built a RAG pipeline from scratch. Not bad for Day 1, right?

### What you accomplished today:

- ✅ Explored a real document and understood its structure
- ✅ Preprocessed text (and learned when NOT to clean too aggressively)
- ✅ Chunked documents using multiple strategies and compared them
- ✅ Created embeddings and verified they capture semantic meaning
- ✅ Built similarity search from scratch with numpy
- ✅ Used a vector database (ChromaDB) with metadata filtering
- ✅ Assembled a complete retrieval pipeline
- ✅ Stress-tested your system and found its weaknesses

### Coming up in Day 2:

Tomorrow we'll add the **intelligence layer**: reranking, query transformation, prompt engineering, hallucination handling, and **actually connecting an LLM** to get real answers.

We'll also learn how to **evaluate** whether your RAG system is actually good or not.

See you tomorrow! 🚀